In [14]:
# ============================================================
# EMPLOYEE ATTRITION — STEP 3: Model Training
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score)
from imblearn.over_sampling import SMOTE

# Set working directory
os.chdir(r"C:\Users\deept\OneDrive\Desktop\Employee Attrition Project")
print("Working directory:", os.getcwd())
print("✅ Libraries loaded")

Working directory: C:\Users\deept\OneDrive\Desktop\Employee Attrition Project
✅ Libraries loaded


In [15]:
# ── 0. Load & Preprocess Data ────────────────────────────────
train_raw = pd.read_csv("train.csv")
test_raw  = pd.read_csv("test.csv")

def preprocess(df, is_train=True):
    df = df.copy()
    df.drop(columns=["Employee ID"], inplace=True)

    if is_train:
        df["Attrition"] = (df["Attrition"] == "Left").astype(int)

    ordinal_mappings = {
        "Work-Life Balance":       {"Poor":1,"Fair":2,"Good":3,"Excellent":4},
        "Job Satisfaction":        {"Low":1,"Medium":2,"High":3,"Very High":4},
        "Performance Rating":      {"Low":1,"Below Average":2,"Average":3,"High":4,"Very High":5},
        "Education Level":         {"High School":1,"Associate Degree":2,
                                    "Bachelor's Degree":3,"Master's Degree":4,"PhD":5},
        "Job Level":               {"Entry":1,"Mid":2,"Senior":3,"Manager":4,"Director":5},
        "Company Size":            {"Small":1,"Medium":2,"Large":3},
        "Company Reputation":      {"Poor":1,"Fair":2,"Good":3,"Excellent":4},
        "Employee Recognition":    {"Low":1,"Medium":2,"High":3,"Very High":4},
    }
    binary_mappings = {
        "Gender":                   {"Male":0,"Female":1},
        "Overtime":                 {"No":0,"Yes":1},
        "Remote Work":              {"No":0,"Yes":1},
        "Leadership Opportunities": {"No":0,"Yes":1},
        "Innovation Opportunities": {"No":0,"Yes":1},
    }

    for col, m in ordinal_mappings.items():
        df[col] = df[col].map(m)
    for col, m in binary_mappings.items():
        df[col] = df[col].map(m)

    df = pd.get_dummies(df, columns=["Job Role","Marital Status"], drop_first=True)

    # Engineered features
    df["Income_per_Tenure_Year"] = df["Monthly Income"] / (df["Years at Company"] + 1)
    df["Promotion_Deprivation"]  = df["Years at Company"] - (df["Number of Promotions"] * 3)
    df["Wellbeing_Score"]        = df["Job Satisfaction"] + df["Work-Life Balance"] + df["Employee Recognition"]
    df["Stress_Score"]           = df["Overtime"] + (df["Distance from Home"] / df["Distance from Home"].max())

    return df

train = preprocess(train_raw, is_train=True)
test  = preprocess(test_raw,  is_train=False)
train, test = train.align(test, join="left", axis=1, fill_value=0)

X = train.drop(columns=["Attrition"])
y = train["Attrition"]

print(f"X shape: {X.shape} | Attrition rate: {y.mean()*100:.1f}%")

X shape: (59598, 30) | Attrition rate: 47.5%


In [16]:
# ── 1. Fill any NaNs from encoding ───────────────────────────
# Some values may not match mappings exactly → results in NaN
# Fill with median — safe and standard practice
print("NaNs in X before fix:", X.isnull().sum().sum())
X = X.fillna(X.median())
print("NaNs in X after fix :", X.isnull().sum().sum())

NaNs in X before fix: 29846
NaNs in X after fix : 0


In [17]:
# ── 2. Train / Validation Split ──────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Val: {X_val.shape}")

Train: (47678, 30) | Val: (11920, 30)


In [18]:
# ── 3. Handle Class Imbalance with SMOTE ─────────────────────
# Creates synthetic minority samples so model doesn't
# just learn to predict "Stayed" for everyone
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After  SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())

Before SMOTE: {0: 25008, 1: 22670}
After  SMOTE: {0: 25008, 1: 25008}


In [19]:
# ── 4. Scale Features ────────────────────────────────────────
# Required for Logistic Regression
# Random Forest doesn't need it but doesn't hurt
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_val_sc   = scaler.transform(X_val)

In [ ]:
# ── 5. Model 1: Logistic Regression ──────────────────────────
# Why: fast, interpretable baseline
# Tells HR exactly which factors drive attrition via coefficients
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_bal)

lr_preds = lr.predict(X_val_sc)
lr_proba = lr.predict_proba(X_val_sc)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_val, lr_preds, target_names=["Stayed","Left"]))
print(f"ROC-AUC : {roc_auc_score(y_val, lr_proba):.4f}")
print(f"F1 (Left): {f1_score(y_val, lr_preds):.4f}")